# Enterprise AI Facility Auditing — Kaggle Notebook

**3-Stage AI Pipeline + Flask API via Ngrok**

| Stage | What runs | Where |
|---|---|---|
| Stage 1 — VLM Inspector | Qwen2.5-VL-7B-Instruct (4-bit NF4) | Locally on T4 GPU |
| Stage 2 — Scene Map Builder | `gemini-2.5-flash` | Google Generative AI API |
| Stage 3 — Final Judge LLM | `google/gemma-3-4b-it` | HuggingFace Inference API |

Run cells **in order**. A public Ngrok URL will be printed at the end.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Install dependencies & import libraries
# ═══════════════════════════════════════════════════════════════════════════════
# Why each package:
#   transformers / accelerate / bitsandbytes  — load Qwen VLM in 4-bit on T4
#   qwen-vl-utils                             — official Qwen VL image helpers
#   google-generativeai                       — Gemini 2.5 Flash (Stage 2)
#   huggingface_hub                           — InferenceClient for Stage 3
#   flask / flask-cors                        — REST API server
#   pyngrok                                   — expose Flask over the internet
#   pillow / requests                         — image download & processing

!pip install -q \
    transformers>=4.45.0 \
    accelerate>=0.33.0 \
    bitsandbytes>=0.43.0 \
    qwen-vl-utils \
    google-generativeai>=0.8.0 \
    "huggingface_hub[inference]>=0.24.0" \
    flask \
    flask-cors \
    pyngrok \
    pillow \
    requests

print('✅ All packages installed.')

In [ ]:
# ─── Standard library ───────────────────────────────────────────────────────
import os, re, json, textwrap, threading, logging
from io import BytesIO

# ─── HTTP / image ────────────────────────────────────────────────────────────
import requests
from PIL import Image

# ─── HuggingFace / Transformers ──────────────────────────────────────────────
import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info
from huggingface_hub import InferenceClient, login

# ─── Google Generative AI ────────────────────────────────────────────────────
import google.generativeai as genai

# ─── Flask ───────────────────────────────────────────────────────────────────
from flask import Flask, request, jsonify
from flask_cors import CORS

# ─── Ngrok ───────────────────────────────────────────────────────────────────
from pyngrok import ngrok, conf

logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)
logging.getLogger('huggingface_hub').setLevel(logging.WARNING)
print('✅ Imports complete.')

---
## 🔑 API Keys
Set your secrets here **before** running the remaining cells.

On Kaggle you can add secrets via **Add-ons → Secrets** and read them with
`from kaggle_secrets import UserSecretsClient` using keys: `GEMINI_API_KEY`, `HF_TOKEN`, `NGROK_AUTH_TOKEN`.

Alternatively, paste them directly into the strings below (do **not** commit to git).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# API Keys (read from Kaggle Secrets or environment variables)
# ═══════════════════════════════════════════════════════════════════════════════

# Option A: Kaggle Secrets (recommended — keeps keys out of the notebook source)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret('GEMINI_API_KEY')
    secret_value_1 = user_secrets.get_secret('HF_TOKEN')
    secret_value_2 = user_secrets.get_secret('NGROK_AUTH_TOKEN')
    GOOGLE_API_KEY   = secret_value_0
    HF_API_TOKEN     = secret_value_1
    NGROK_AUTH_TOKEN = secret_value_2
    print('✅ Keys loaded from Kaggle Secrets.')
except Exception:
    # Option B: Environment variables / manual override
    GOOGLE_API_KEY   = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY', 'YOUR_GOOGLE_API_KEY_HERE')
    HF_API_TOKEN     = os.environ.get('HF_TOKEN') or os.environ.get('HF_API_TOKEN', 'YOUR_HF_TOKEN_HERE')
    NGROK_AUTH_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', 'YOUR_NGROK_TOKEN_HERE')
    print('⚠️  Keys loaded from environment variables (or hardcoded placeholders).')

# Validate
for name, val in [('GOOGLE_API_KEY', GOOGLE_API_KEY),
                   ('HF_API_TOKEN',   HF_API_TOKEN),
                   ('NGROK_AUTH_TOKEN', NGROK_AUTH_TOKEN)]:
    if not val or val.startswith('YOUR_'):
        print(f'⚠️  {name} is not set — the corresponding stage will fail.')
    else:
        print(f'   {name}: {'*' * 6}{val[-4:]}')

# Hugging Face download reliability settings (important on Kaggle)
os.environ.setdefault('HF_HOME', '/kaggle/working/.hf_home')
os.environ.setdefault('TRANSFORMERS_CACHE', '/kaggle/working/.hf_home/transformers')
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '120')
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '30')

if HF_API_TOKEN and not HF_API_TOKEN.startswith('YOUR_'):
    os.environ['HF_TOKEN'] = HF_API_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_API_TOKEN
    try:
        login(token=HF_API_TOKEN, add_to_git_credential=False)
        print('✅ Hugging Face login ready (token attached for model download).')
    except Exception as exc:
        print(f'⚠️  Hugging Face login warning: {exc}')
else:
    print('⚠️  HF_API_TOKEN missing — large model downloads may be blocked/rate-limited.')

HF_AUTH_TOKEN = HF_API_TOKEN if HF_API_TOKEN and not HF_API_TOKEN.startswith('YOUR_') else None
print(f"   HF cache dir: {os.environ['TRANSFORMERS_CACHE']}")


---
## Stage 1 — Qwen2.5-VL-7B-Instruct loaded in 4-bit (NF4)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Load Qwen2.5-VL-7B-Instruct via AutoModelForImageTextToText (4-bit NF4)
#
# Why 4-bit NF4?
#   The T4 has only 16 GB VRAM. The 7B model in fp16 needs ~14 GB just for
#   weights, leaving almost no room for activations. NF4 shrinks it to ~4–5 GB.
#
# max_pixels = 512 * 28 * 28 = 401 408
#   This caps the number of visual tokens the processor extracts, preventing
#   OOM when high-resolution facility images are passed in.
# ═══════════════════════════════════════════════════════════════════════════════

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # saves an extra ~0.4 bits per parameter
)

print(f'Loading {MODEL_ID} in 4-bit NF4 — first run may take ~2–4 min (or longer while shards download) …')

qwen_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',          # puts layers on GPU automatically
    torch_dtype=torch.float16,
    trust_remote_code=True,
    token=HF_AUTH_TOKEN,
    cache_dir=os.environ.get('TRANSFORMERS_CACHE'),
)
qwen_model.eval()

qwen_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    max_pixels=512 * 28 * 28,   # cap visual tokens ← CRITICAL for T4
    trust_remote_code=True,
    token=HF_AUTH_TOKEN,
    cache_dir=os.environ.get('TRANSFORMERS_CACHE'),
)

print('✅ Qwen2.5-VL-7B-Instruct loaded.')
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   VRAM used: {allocated:.1f} GB / {total:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Smoke test for the Qwen VLM
#
# We deliberately avoid Wikipedia/Wikimedia URLs (they return 403 Forbidden
# because they block non-browser user-agents). Instead we use picsum.photos,
# which always serves a public JPEG without authentication.
# ═══════════════════════════════════════════════════════════════════════════════

SMOKE_TEST_URL = 'https://picsum.photos/id/237/800/600.jpg'


def _download_image(url: str) -> Image.Image:
    """Download an image from a URL and return a PIL Image.
    Supports http/https URLs and local file:// paths.
    """
    if url.startswith('file://'):
        local_path = url[len('file://'):]
        return Image.open(local_path).convert('RGB')

    headers = {'User-Agent': 'Mozilla/5.0 (compatible; AuditBot/1.0)'}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert('RGB')


def run_qwen_stage1(image_urls: list[str], question_text: str) -> dict:
    """
    Stage 1 — VLM Inspector.

    Sends all images to Qwen2.5-VL and asks it to extract structured visual
    facts relevant to *question_text*.

    Returns a dict with keys:
        objects_present      : list[str]
        issues_found         : list[str]
        condition_observations: list[str]
    """
    # Build the message: interleave images then ask the question
    content = []
    for url in image_urls:
        content.append({'type': 'image', 'image': url})

    system_prompt = textwrap.dedent("""
        You are a meticulous facility auditing inspector.
        Examine the provided images carefully and extract ONLY observable visual facts.
        Return your answer as valid JSON with exactly these keys:
          - objects_present       (list of strings)
          - issues_found          (list of strings, empty list if none)
          - condition_observations(list of strings)
        Do NOT include any text outside the JSON object.
    """).strip()

    content.append({'type': 'text', 'text': f'Question context: {question_text}'})

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': content},
    ]

    # Apply the chat template and prepare pixel tensors
    text_input = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        return_tensors='pt',
        padding=True,
    ).to(qwen_model.device)

    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )

    # Decode only the newly generated tokens (exclude the input prompt)
    generated_ids = [
        out[len(inp):]
        for inp, out in zip(inputs.input_ids, output_ids)
    ]
    raw_text = qwen_processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    return clean_and_parse_json(raw_text)


# ── Run the smoke test ──────────────────────────────────────────────────────
print('Running Stage 1 smoke test …')
smoke_result = run_qwen_stage1(
    image_urls=[SMOKE_TEST_URL],
    question_text='Describe the objects and overall condition visible in the image.',
)
print('Smoke test result:')
print(json.dumps(smoke_result, indent=2))
print('✅ Stage 1 smoke test passed.' if smoke_result else '⚠️  Stage 1 returned empty result — check the model output above.')

---
## Utility helpers

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Utility helpers
#
# clean_and_parse_json  — strips ```json … ``` fences before json.loads()
#                         so LLM markdown formatting never breaks the pipeline.
# SKIP_QUESTION_IDS     — physical/manual questions that cannot be scored by AI.
# ═══════════════════════════════════════════════════════════════════════════════

# Questions the AI pipeline must skip (physical or manual checks)
SKIP_QUESTION_IDS: set[str] = {'Q004', 'Q005'}


def clean_and_parse_json(raw: str) -> dict | list:
    """
    Robustly parse JSON from an LLM response that may be wrapped in
    ```json ... ``` or ``` ... ``` markdown code fences.

    Steps:
    1. Strip leading/trailing whitespace.
    2. Remove markdown fences (```json or ```).
    3. Extract the first {...} or [...] block with a regex as a last resort.
    4. Call json.loads().

    Raises json.JSONDecodeError if the text cannot be parsed.
    """
    text = raw.strip()

    # Step 2 — remove ```json / ``` fences
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'```\s*$',          '', text, flags=re.MULTILINE)
    text = text.strip()

    # Step 3 — extract first JSON object or array if there is surrounding prose
    match = re.search(r'(\{.*\}|\[.*\])', text, re.DOTALL)
    if match:
        text = match.group(1)

    return json.loads(text)


def safe_parse_json(raw: str, fallback: dict | None = None) -> dict:
    """Like clean_and_parse_json but returns *fallback* on failure instead of raising."""
    try:
        return clean_and_parse_json(raw)
    except Exception as exc:
        logging.warning('JSON parse error: %s — raw snippet: %.120s', exc, raw)
        return fallback if fallback is not None else {}


print('✅ Helper utilities defined.')

---
## Stage 2 — Scene Map Builder (Gemini 2.5 Flash)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Stage 2: Cognitive Scene Map via Gemini 2.5 Flash
#
# Receives the list of Stage-1 fact dicts (one per image) and synthesises
# them into a single descriptive paragraph: the "Cognitive Scene Map".
# This keeps the heavy text reasoning away from the GPU.
# ═══════════════════════════════════════════════════════════════════════════════

genai.configure(api_key=GOOGLE_API_KEY)
_gemini_model = genai.GenerativeModel('gemini-2.5-flash')


def run_gemini_stage2(all_stage1_facts: list[dict]) -> str:
    """
    Stage 2 — Scene Map Builder.

    Parameters
    ----------
    all_stage1_facts : list[dict]
        Each element is the dict returned by run_qwen_stage1() for one image.

    Returns
    -------
    str
        A single cohesive paragraph describing the entire facility layout
        and overall condition (the "Cognitive Scene Map").
    """
    facts_json = json.dumps(all_stage1_facts, indent=2)

    prompt = textwrap.dedent(f"""
        You are an expert facility auditor writing an executive summary.

        Below are structured visual facts extracted from multiple facility images
        by an AI inspector. Each fact block covers one image.

        VISUAL FACTS:
        {facts_json}

        Write ONE cohesive paragraph (the "Cognitive Scene Map") that:
        - Describes the overall facility layout and spatial arrangement.
        - Summarises the general condition (cleanliness, equipment state, hazards).
        - Highlights the most significant issues found.
        - Uses professional audit language.
        - Does NOT include headings, bullet points, or JSON — plain prose only.
    """).strip()

    response = _gemini_model.generate_content(prompt)
    return response.text.strip()


print('✅ Stage 2 (Gemini 2.5 Flash) ready.')

---
## Stage 3 — Final Judge LLM (HuggingFace Inference API)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Stage 3: Final Judge via HuggingFace InferenceClient
#
# Model : google/gemma-3-4b-it
#   (Swap to e.g. mistralai/Mistral-7B-Instruct-v0.3 if quota is tight.)
#
# The judge receives:
#   • The Cognitive Scene Map (Stage 2)
#   • The raw visual facts for the current question (Stage 1)
#   • The question text and its 5-level scoring rubric
# and returns JSON: {"score": 1-5, "observation": "...", "confidence": "high|medium|low"}
# ═══════════════════════════════════════════════════════════════════════════════

HF_JUDGE_MODEL = 'google/gemma-3-4b-it'

_hf_client = InferenceClient(
    model=HF_JUDGE_MODEL,
    token=HF_AUTH_TOKEN,
)


def run_hf_stage3(
    scene_map: str,
    visual_facts: dict,
    question: dict,
) -> dict:
    """
    Stage 3 — Final Judge LLM.

    Parameters
    ----------
    scene_map    : str   — Cognitive Scene Map paragraph from Stage 2.
    visual_facts : dict  — Stage 1 facts for this specific question.
    question     : dict  — Question object from the frontend, expected keys:
                           questionId, questionText,
                           score1 … score5  (rubric descriptions).

    Returns
    -------
    dict with keys: score (int 1-5), observation (str), confidence (str)
    """
    # Build the rubric block
    rubric_lines = '\n'.join(
        f'  Score {i}: {question.get(f"score{i}", "N/A")}' for i in range(1, 6)
    )

    prompt = textwrap.dedent(f"""
        You are a strict facility audit judge. Your task is to score a single
        audit question from 1 to 5 based on the evidence provided.

        === COGNITIVE SCENE MAP ===
        {scene_map}

        === VISUAL FACTS FOR THIS QUESTION ===
        {json.dumps(visual_facts, indent=2)}

        === QUESTION ===
        ID  : {question.get('questionId', 'N/A')}
        Text: {question.get('questionText', 'N/A')}

        === SCORING RUBRIC ===
        {rubric_lines}

        INSTRUCTIONS:
        - Choose the score that best matches the evidence.
        - Write a single concise observation sentence (max 25 words).
        - Assign a confidence: "high", "medium", or "low".
        - Return ONLY a valid JSON object — no markdown, no prose outside the JSON:
          {{"score": <int 1-5>, "observation": "<string>", "confidence": "<high|medium|low>"}}
    """).strip()

    response = _hf_client.chat_completion(
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=256,
        temperature=0.1,   # near-deterministic for consistent scoring
    )

    raw_text = response.choices[0].message.content
    parsed   = safe_parse_json(raw_text, fallback={'score': 1, 'observation': raw_text[:200], 'confidence': 'low'})

    # Validate / coerce types
    parsed['score'] = max(1, min(5, int(parsed.get('score', 1))))
    parsed.setdefault('observation', '')
    parsed.setdefault('confidence',  'low')
    return parsed


print('✅ Stage 3 (HuggingFace InferenceClient) ready.')

---
## Flask API — `/score` endpoint

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Flask application with /score endpoint
#
# Expected request body (JSON) from Google Apps Script:
# {
#   "audit_id"  : "123",
#   "image_urls": ["url1", "url2"],
#   "questions" : [
#     {
#       "questionId"  : "Q001",
#       "questionText": "Is the fire exit clearly marked?",
#       "score1": "No signage visible",
#       "score2": "Partially visible",
#       "score3": "Visible but obstructed",
#       "score4": "Clearly visible",
#       "score5": "Clearly visible with lighting"
#     },
#     ...
#   ]
# }
#
# Response (JSON):
# {
#   "success"   : true,
#   "audit_id"  : "123",
#   "scene_map" : "<Stage 2 paragraph>",
#   "results"   : [
#     {"questionId": "Q001", "score": 4, "observation": "...", "confidence": "high"},
#     {"questionId": "Q004", "skipped": true, "reason": "manual check"},
#     ...
#   ]
# }
# ═══════════════════════════════════════════════════════════════════════════════

app = Flask(__name__)
CORS(app)  # allow cross-origin requests from Google Apps Script


@app.route('/health', methods=['GET'])
def health_check():
    """Simple liveness probe so the frontend can verify the API is up."""
    return jsonify({'status': 'ok', 'model': MODEL_ID})


@app.route('/score', methods=['POST'])
def score_endpoint():
    """
    Main scoring endpoint.

    Runs the full 3-stage pipeline for every non-skipped question and
    returns structured JSON results to the Apps Script frontend.

    All exceptions are caught and returned as {"success": false, "error": "..."},
    so the server never crashes on a bad request.
    """
    try:
        payload = request.get_json(force=True, silent=True)
        if not payload:
            return jsonify({'success': False, 'error': 'Request body is not valid JSON.'}), 400

        audit_id   = payload.get('audit_id', 'unknown')
        image_urls = payload.get('image_urls', [])
        questions  = payload.get('questions',  [])

        if not image_urls:
            return jsonify({'success': False, 'error': '"image_urls" list is empty.'}), 400
        if not questions:
            return jsonify({'success': False, 'error': '"questions" list is empty.'}), 400

        logging.info('audit_id=%s  images=%d  questions=%d', audit_id, len(image_urls), len(questions))

        # ── Stage 1: run Qwen once per image to collect all visual facts ──────
        # We collect facts for each image independently so Stage 2 gets
        # per-image granularity for the scene map.
        all_facts: list[dict] = []
        for img_url in image_urls:
            try:
                facts = run_qwen_stage1(
                    image_urls=[img_url],
                    question_text='General facility inspection',
                )
                facts['_source_url'] = img_url  # keep provenance
                all_facts.append(facts)
            except Exception as exc:
                logging.warning('Stage 1 failed for %s: %s', img_url, exc)
                all_facts.append({
                    '_source_url':          img_url,
                    'objects_present':       [],
                    'issues_found':          [f'Image load/inference error: {exc}'],
                    'condition_observations': [],
                })

        # ── Stage 2: build the Cognitive Scene Map from all Stage-1 facts ─────
        try:
            scene_map = run_gemini_stage2(all_facts)
        except Exception as exc:
            logging.error('Stage 2 failed: %s', exc)
            scene_map = f'Scene map unavailable ({exc}).'

        # ── Stage 3: score each question ──────────────────────────────────────
        results: list[dict] = []
        for question in questions:
            q_id = question.get('questionId', '')

            # Skip physical / manual questions
            if q_id in SKIP_QUESTION_IDS:
                results.append({
                    'questionId': q_id,
                    'skipped':    True,
                    'reason':     'manual check — requires physical inspection',
                })
                logging.info('Skipping %s (manual check).', q_id)
                continue

            # Run Stage 1 again, this time focused on the specific question
            try:
                question_facts = run_qwen_stage1(
                    image_urls=image_urls,
                    question_text=question.get('questionText', ''),
                )
            except Exception as exc:
                logging.warning('Stage 1 (question-focused) failed for %s: %s', q_id, exc)
                question_facts = {}

            # Run Stage 3
            try:
                judgment = run_hf_stage3(scene_map, question_facts, question)
                results.append({'questionId': q_id, **judgment})
            except Exception as exc:
                logging.error('Stage 3 failed for %s: %s', q_id, exc)
                results.append({
                    'questionId':   q_id,
                    'score':        1,
                    'observation':  f'Scoring error: {exc}',
                    'confidence':   'low',
                })

        return jsonify({
            'success':   True,
            'audit_id':  audit_id,
            'scene_map': scene_map,
            'results':   results,
        })

    except Exception as exc:   # last-resort safety net
        logging.exception('Unhandled exception in /score')
        return jsonify({'success': False, 'error': str(exc)}), 500


print('✅ Flask app defined with /health and /score endpoints.')

---
## Launch Flask in a background thread + expose via Ngrok

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Start the Flask server in a background thread, then expose
#           it to the internet via Ngrok.
#
# Why a background thread?
#   app.run() is blocking. Running it in a daemon thread lets the Kaggle
#   notebook kernel stay alive and print the public URL in the main thread.
#
# IMPORTANT: Paste your Ngrok auth token into Cell 2 (or Kaggle Secrets).
#   Free Ngrok accounts allow one active tunnel per session.
# ═══════════════════════════════════════════════════════════════════════════════

FLASK_PORT = 5000

# ── Ngrok authentication ─────────────────────────────────────────────────────
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# ── Kill any leftover Ngrok tunnels from a previous run ───────────────────────
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# ── Start Flask in a daemon thread ────────────────────────────────────────────
flask_thread = threading.Thread(
    target=lambda: app.run(
        host='0.0.0.0',
        port=FLASK_PORT,
        debug=False,    # debug=True spawns a reloader that conflicts with threads
        use_reloader=False,
    ),
    daemon=True,        # thread dies automatically when the notebook kernel stops
)
flask_thread.start()

import time
time.sleep(2)  # give Flask a moment to bind to the port

# ── Open Ngrok tunnel ────────────────────────────────────────────────────────
tunnel     = ngrok.connect(FLASK_PORT, 'http')
public_url = tunnel.public_url

print('=' * 60)
print(f'✅ Flask API is running on port {FLASK_PORT}')
print(f'🌐 Public Ngrok URL : {public_url}')
print(f'   Health check     : {public_url}/health')
print(f'   Score endpoint   : {public_url}/score  (POST)')
print('=' * 60)
print()
print('Copy the Ngrok URL above into your Google Apps Script')
print('as the BASE_URL constant.')